In [1]:
# =====================================================================
# CELL 1 — Install
# =====================================================================
!pip install -U "transformers>=4.48.0" accelerate bitsandbytes qwen-vl-utils datasets tqdm -q

import os
from google.colab import drive
drive.mount('/content/drive')

# Model goes to temp disk (redownloads in 2-3 min, avoids storage quota issues)
os.environ["HF_HOME"] = "/content/model_cache"
os.environ["HF_DATASETS_CACHE"] = "/content/dataset_cache"

# Only tiny checkpoint JSONs go to Drive (a few KB each)
RESULTS_ROOT = "/content/drive/MyDrive/gsv_math_results"
os.makedirs(RESULTS_ROOT, exist_ok=True)

import torch
print(f"\nCUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 122.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 45.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.2/80.2 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.5/35.5 MB 19.3 MB/s eta 0:00:00
Mounted at /content/drive

CUDA available: True
GPU: Tesla T4
VRAM: 15.6 GB


In [2]:
# =====================================================================
# CELL 2 — Improved Parsing Logic v2
# =====================================================================
import re

FINAL_ANSWER_PATTERNS = [
    r'\\boxed\{([^}]*)\}',
    r'[Ff]inal\s*[Aa]nswer\s*[:\-]?\s*(.{1,80})',
    r'[Tt]herefore[,\s]+(?:the\s+)?(?:answer|value|result)\s+is\s*[:\-]?\s*(.{1,80})',
    r'[Tt]he\s+answer\s+is\s*[:\-]?\s*(.{1,80})',
    r'[Ss]o\s+the\s+answer\s+is\s*[:\-]?\s*(.{1,80})',
    r'=\s*(\S+)\s*$',
]

def extract_final_answer_region(raw_text, tail_chars=300):
    for pattern in FINAL_ANSWER_PATTERNS:
        matches = list(re.finditer(pattern, raw_text, re.IGNORECASE | re.DOTALL))
        if matches:
            return matches[-1].group(1).strip()
    return raw_text[-tail_chars:] if len(raw_text) > tail_chars else raw_text

def clean_free_form(text):
    if not isinstance(text, str): return str(text)
    text = text.strip().lower()
    prefixes = ["the answer is", "therefore, the answer is", "so the answer is",
                "the value is", "answer is", "value is", "equals", "it is",
                "the final answer is", "final answer:", "answer:"]
    for prefix in prefixes:
        if text.startswith(prefix):
            text = text[len(prefix):].strip()
    match = re.match(r'^[a-zA-Z\s]+=\s*(.*)$', text)
    if match: text = match.group(1).strip()
    return text.rstrip('.!?*, ')

def get_most_similar(extraction, choices):
    distances = [-len(set(extraction.lower()) & set(choice.lower())) for choice in choices]
    return choices[distances.index(min(distances))]

def normalize_extracted_answer(extraction, choices, question_type, answer_type, precision=2):
    extraction = str(extraction).strip() if extraction else ""
    extraction = extract_final_answer_region(extraction)
    if question_type == 'multi_choice':
        letter = re.findall(r'\(([a-zA-Z])\)', extraction)
        extraction = letter[0].upper() if letter else extraction
        options = [chr(ord('A') + i) for i in range(len(choices))]
        if extraction in options:
            extraction = choices[options.index(extraction)]
        else:
            extraction = get_most_similar(clean_free_form(extraction), choices)
    else:
        cleaned = clean_free_form(extraction)
        if answer_type in ['integer', 'float']:
            numbers = re.findall(r'-?\d+\.?\d*', cleaned)
            extraction = numbers[-1] if numbers else cleaned
    return extraction

def is_correct(pred, gt, answer_type):
    if pred.lower().strip() == gt.lower().strip(): return 1
    if answer_type in ['integer', 'float']:
        try:
            if abs(float(pred) - float(gt)) < 1e-5: return 1
        except: pass
    return 0

In [3]:
# =====================================================================
# CELL 3 — Config (FEW-SHOT)
# =====================================================================
MODEL_ID = "Qwen/Qwen2.5-VL-7B-Instruct"
MAX_NEW_TOKENS = 512

RUN_NAME = "qwen25vl7b_fewshot_4exemplars"
BATCH_SIZE = 50

RUN_DIR = f"{RESULTS_ROOT}/{RUN_NAME}"
import os
os.makedirs(RUN_DIR, exist_ok=True)

print(f"Target Model: {MODEL_ID}")
print(f"Run name:     {RUN_NAME}")
print(f"Checkpoints:  {RUN_DIR}")

Target Model: Qwen/Qwen2.5-VL-7B-Instruct
Run name:     qwen25vl7b_fewshot_4exemplars
Checkpoints:  /content/drive/MyDrive/gsv_math_results/qwen25vl7b_fewshot_4exemplars


In [4]:
# =====================================================================
# CELL 4 — Load Data, Select Few-Shot Exemplars, Load Model
# =====================================================================
from datasets import load_dataset
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
import torch

# Load the EVALUATION set (testmini = 1000 samples)
print("Loading MathVista testmini (evaluation)...")
mathvista = load_dataset("AI4Math/MathVista", split="testmini")
print(f"Loaded {len(mathvista)} evaluation samples.")

# Load the TEST set to pick exemplars (NEVER use testmini for exemplars!)
print("\nLoading MathVista test split (for exemplar selection)...")
mathvista_test = load_dataset("AI4Math/MathVista", split="test")
print(f"Loaded {len(mathvista_test)} test samples.")

# --- Check what types exist ---
from collections import Counter
type_counts = Counter((s["question_type"], s["answer_type"]) for s in mathvista_test if s["answer"].strip())
print("\nAvailable (question_type, answer_type) combos with non-empty answers:")
for k, v in type_counts.most_common():
    print(f"  {k}: {v} samples")

# --- Select 4 diverse exemplars with non-empty answers ---
seen_types = set()
EXEMPLARS = []

for sample in mathvista_test:
    if not sample["answer"].strip():
        continue
    key = (sample["question_type"], sample["answer_type"])
    if key not in seen_types:
        seen_types.add(key)
        EXEMPLARS.append(sample)
    if len(EXEMPLARS) >= 4:
        break

print(f"\nSelected {len(EXEMPLARS)} exemplars:")
for i, ex in enumerate(EXEMPLARS):
    print(f"  [{i+1}] type={ex['question_type']}/{ex['answer_type']}  answer='{ex['answer']}'")
    print(f"       query: {ex['query'][:80]}...")

# --- Build the few-shot conversation history (text-only, saves VRAM) ---
FEW_SHOT_MESSAGES = []
for ex in EXEMPLARS:
    FEW_SHOT_MESSAGES.append({
        "role": "user",
        "content": [{"type": "text", "text": ex["query"]}]
    })
    FEW_SHOT_MESSAGES.append({
        "role": "assistant",
        "content": [{"type": "text", "text": str(ex["answer"])}]
    })

print(f"\nFew-shot history: {len(FEW_SHOT_MESSAGES)} messages ({len(EXEMPLARS)} Q&A pairs)")

# --- Load model ---
print(f"\nLoading model: {MODEL_ID} in 4-bit...")
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID, quantization_config=quantization_config, device_map="auto"
)
processor = AutoProcessor.from_pretrained(MODEL_ID)
print("Model and Processor loaded successfully.")

Loading MathVista testmini (evaluation)...


README.md:   0%|          | 0.00/11.6k [00:00<?, ?B/s]

data/testmini-00000-of-00001-725687bf7a1(…): reconstructing file:   0%|          |  0.00B /  142MB            

data/testmini-00000-of-00001-725687bf7a1(…): downloading bytes:           |  0.00B            

data/test-00000-of-00002-6b81bd7f7e2065e(…): reconstructing file:   0%|          |  0.00B /  358MB            

...

tokenizer_config.json:   0%|          | 0.00/5.70k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Model and Processor loaded successfully.


In [5]:
# =====================================================================
# CELL 5 — Few-Shot Evaluation Loop (with batch checkpointing)
# =====================================================================
import gc, json
from pathlib import Path
from tqdm import tqdm
from qwen_vl_utils import process_vision_info
import torch

RESOLUTION_LADDER = [1003520, 501760, 313600]

def load_completed_pids():
    done = set()
    for f in Path(RUN_DIR).glob("batch_*.json"):
        try:
            with open(f) as fh:
                batch = json.load(fh)
            done.update(item["question_id"] for item in batch)
        except json.JSONDecodeError:
            print(f"  [warn] {f.name} incomplete, skipping.")
    return done

def next_batch_index():
    existing = list(Path(RUN_DIR).glob("batch_*.json"))
    if not existing: return 0
    return max(int(f.stem.split("_")[1]) for f in existing) + 1

def run_fewshot_inference(sample):
    """Run inference with few-shot exemplars prepended, progressive resolution."""
    for max_px in RESOLUTION_LADDER:
        try:
            # Build the full message list: few-shot history + actual test question
            messages = FEW_SHOT_MESSAGES + [
                {
                    "role": "user",
                    "content": [
                        {"type": "image", "image": sample["decoded_image"], "max_pixels": max_px},
                        {"type": "text", "text": sample["query"]}
                    ],
                }
            ]

            text_prompt = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
            image_inputs, video_inputs = process_vision_info(messages)

            inputs = processor(
                text=[text_prompt], images=image_inputs, padding=True, return_tensors="pt"
            ).to(model.device)

            with torch.no_grad():
                output_ids = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS)

            generated_ids = [output_ids[j][len(inputs.input_ids[j]):] for j in range(len(output_ids))]
            raw_answer = processor.batch_decode(generated_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0]

            del inputs, output_ids, generated_ids, messages, text_prompt, image_inputs
            gc.collect()
            torch.cuda.empty_cache()
            return raw_answer

        except torch.cuda.OutOfMemoryError:
            gc.collect()
            torch.cuda.empty_cache()
            if max_px == RESOLUTION_LADDER[-1]:
                return "[OOM_SKIP]"
            continue
    return "[OOM_SKIP]"

# --- Main Loop ---
completed_pids = load_completed_pids()
print(f"Resuming run '{RUN_NAME}': {len(completed_pids)}/{len(mathvista)} samples already done.")

remaining = [s for s in mathvista if s["pid"] not in completed_pids]
print(f"{len(remaining)} samples left to run.\n")

batch_idx = next_batch_index()
batch_results = []

for i, sample in enumerate(tqdm(remaining, desc="Few-Shot Evaluating")):
    raw_answer = run_fewshot_inference(sample)

    if raw_answer == "[OOM_SKIP]":
        correct_flag = 0
        normalized_pred = "[OOM]"
    else:
        normalized_pred = normalize_extracted_answer(
            raw_answer, sample.get("choices", []), sample["question_type"], sample["answer_type"]
        )
        correct_flag = is_correct(normalized_pred, sample["answer"], sample["answer_type"])

    batch_results.append({
        "question_id": sample["pid"],
        "skills": sample["metadata"]["skills"],
        "correct": correct_flag,
        "raw_answer": raw_answer
    })

    if len(batch_results) >= BATCH_SIZE or i == len(remaining) - 1:
        out_file = f"{RUN_DIR}/batch_{batch_idx:04d}.json"
        with open(out_file, "w") as fh:
            json.dump(batch_results, fh)
        print(f"  [checkpoint] saved {out_file}  ({len(batch_results)} samples)")
        batch_idx += 1
        batch_results = []

print("\nBatch complete for this session.")

Resuming run 'qwen25vl7b_fewshot_4exemplars': 950/1000 samples already done.
50 samples left to run.





Few-Shot Evaluating:   0%|          | 0/50 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:944: UserWarning: inner dimension (3420) is not aligned for fast kernel with blocksize=64, falling back to slower implementation.
  warn(


Few-Shot Evaluating:   2%|▏         | 1/50 [00:45<37:16, 45.65s/it]

Few-Shot Evaluating:   4%|▍         | 2/50 [00:58<20:57, 26.21s/it]

Few-Shot Evaluating:   6%|▌         | 3/50 [01:17<18:02, 23.02s/it]

Few-Shot Evaluating:   8%|▊         | 4/50 [01:43<18:27, 24.07s/it]

Few-Shot Evaluating:  10%|█         | 5/50 [02:42<27:41, 36.92s/it]

Few-Shot Evaluating:  12%|█▏        | 6/50 [03:09<24:34, 33.50s/it]

Few-Shot Evaluating:  14%|█▍        | 7/50 [03:36<22:21, 31.20s/it]

Few-Shot Evaluating:  16%|█▌        | 8/50 [04:43<29:54, 42.71s/it]

Few-Shot Evaluating:  18%|█▊        | 9/50 [04:59<23:31, 34.42s/it]

Few-Shot Evaluating:  20%|██        | 10/50 [05:19<19:52, 29.82s/it]

Few-Shot Evaluating:  22%|██▏     

  [checkpoint] saved /content/drive/MyDrive/gsv_math_results/qwen25vl7b_fewshot_4exemplars/batch_0019.json  (50 samples)

Batch complete for this session.


In [6]:
# =====================================================================
# CELL 6 — Aggregate all batches
# =====================================================================
import json, glob

results = []
for f in sorted(glob.glob(f"{RUN_DIR}/batch_*.json")):
    with open(f) as fh:
        results.extend(json.load(fh))

print(f"Aggregated {len(results)} / {len(mathvista)} total samples")

if len(results) >= len(mathvista):
    final_path = f"{RUN_DIR}/FINAL_results.json"
    with open(final_path, "w") as fh:
        json.dump(results, fh)
    print(f"Saved: {final_path}")
else:
    print(f"\nNot complete yet. Re-run Cell 5 to continue.")

Aggregated 1000 / 1000 total samples
Saved: /content/drive/MyDrive/gsv_math_results/qwen25vl7b_fewshot_4exemplars/FINAL_results.json


In [7]:
# =====================================================================
# CELL 7 — Metrics + Comparison with Zero-Shot
# =====================================================================
skill_to_category = {
    "geometry reasoning": "geometry",
    "arithmetic reasoning": "arithmetic",
    "algebraic reasoning": "algebra",
    "logical reasoning": "logic",
    "numeric commonsense": "numeric",
    "scientific reasoning": "scientific",
    "statistical reasoning": "statistical",
}

categories = ["all", "geometry", "arithmetic", "algebra", "logic", "numeric", "scientific", "statistical"]
metrics = {cat: {"correct": 0, "total": 0} for cat in categories}

for res in results:
    correct = res["correct"]
    metrics["all"]["correct"] += correct
    metrics["all"]["total"] += 1
    for skill in res.get("skills", []):
        cat = skill_to_category.get(skill)
        if cat in metrics:
            metrics[cat]["correct"] += correct
            metrics[cat]["total"] += 1

print("\n" + "="*60)
print(f"{'Qwen2.5-VL-7B — Few-Shot (4 exemplars)':^60}")
print("="*60)
print(f"{'Category':<20} | {'Correct':<10} | {'Total':<10} | {'Accuracy':<10}")
print("-"*60)

for cat in categories:
    correct = metrics[cat]["correct"]
    total = metrics[cat]["total"]
    acc = (correct / total * 100) if total > 0 else 0.0
    print(f"{cat.capitalize():<20} | {correct:<10} | {total:<10} | {acc:.2f}%")

print("="*60)

# Compare with zero-shot
print(f"\n{'COMPARISON':^60}")
print("-"*60)
fewshot_acc = metrics['all']['correct'] / metrics['all']['total'] * 100
print(f"  Zero-shot (v2 parser):  61.9%")
print(f"  Few-shot (this run):    {fewshot_acc:.1f}%")
print(f"  Delta:                  {fewshot_acc - 61.9:+.1f}%")
print("="*60)


           Qwen2.5-VL-7B — Few-Shot (4 exemplars)           
Category             | Correct    | Total      | Accuracy  
------------------------------------------------------------
All                  | 621        | 1000       | 62.10%
Geometry             | 136        | 239        | 56.90%
Arithmetic           | 206        | 353        | 58.36%
Algebra              | 160        | 281        | 56.94%
Logic                | 10         | 37         | 27.03%
Numeric              | 56         | 144        | 38.89%
Scientific           | 67         | 122        | 54.92%
Statistical          | 238        | 301        | 79.07%

                         COMPARISON                         
------------------------------------------------------------
  Zero-shot (v2 parser):  61.9%
  Few-shot (this run):    62.1%
  Delta:                  +0.2%
